## Imports

In [2]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    import gdown
    drive.mount('/content/drive')

In [2]:
import io
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from pathlib import Path

## Setup
1) Loading data
2) Constants

In [ ]:
# 1)
# Training file
train_file_id = "1hT51HoLUSZohAl4CRIRopnh0gqnj8Pz4"
train_output = 'train_sampled_esm2_650m.pt'
gdown.download(f'https://drive.google.com/uc?id={train_file_id}', train_output, quiet=False)
train_path = Path(train_output)

# Validation file
val_file_id = "1H0lC_7G0cwr8hB0dMyrWf7tcEUB8oB4e"
val_output = 'val_full_esm2_650m.pt'
gdown.download(f'https://drive.google.com/uc?id={val_file_id}', val_output, quiet=False)
val_path = Path(val_output)

In [4]:
# 2)
# Model Architecture
INPUT_DIM = 1280          # ESM-2 embedding dimension
LATENT_DIM = 256          # Latent space dimensionality
HIDDEN_DIM = 1024         # Hidden layer dimension

BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100
RECONSTRUCTION_WEIGHT = 1.0
KL_WEIGHT = 0.05
PREDICTION_WEIGHT = 1.0
OPTIM_STEPS = 25          # Steps for latent optimization
OPTIM_LR = 0.01           # Learning rate for latent optimization

## Robust Loader

In [5]:
def robust_pt_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except Exception:
        import pickle
        class CPU_Unpickler(pickle.Unpickler):
            def find_class(self, module, name):
                if module == 'torch.storage' and name == '_load_from_bytes':
                    return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
                return super().find_class(module, name)
        with open(path, 'rb') as f:
            return CPU_Unpickler(f).load()

## Dataset Class

In [6]:
class ProteinDataset(Dataset):
    def __init__(self, pt_path, scaler=None):
        print(f"Loading Dataset: {pt_path}")
        data_obj = robust_pt_load(pt_path)

        self.embeddings = torch.from_numpy(np.array(data_obj['seq_embeddings'])).float()
        labels_np = np.array(data_obj['delta_g']).reshape(-1, 1)
        self.clusters = np.array(data_obj['wt_cluster'])
        self.mut_types = np.array(data_obj['mut_type'])

        self.cluster_to_indices = {}
        for idx, c in enumerate(self.clusters):
            if c not in self.cluster_to_indices:
                self.cluster_to_indices[c] = []
            self.cluster_to_indices[c].append(idx)

        self.wt_lookup = {}
        for i in range(len(self.clusters)):
            if str(self.mut_types[i]).lower().strip() == 'wt':
                self.wt_lookup[self.clusters[i]] = {
                    'dg': labels_np[i][0],
                    'emb': self.embeddings[i],
                    'idx': i
                }

        if scaler is None:
            self.scaler = StandardScaler()
            self.labels_scaled = self.scaler.fit_transform(labels_np)
        else:
            self.scaler = scaler
            self.labels_scaled = self.scaler.transform(labels_np)

        self.labels_scaled = torch.from_numpy(self.labels_scaled).float()
        self.raw_labels = torch.from_numpy(labels_np).float()

    def __len__(self):
        return self.embeddings.shape[0]

    def __getitem__(self, idx):
        cluster_id = self.clusters[idx]
        wt_info = self.wt_lookup.get(cluster_id, None)
        wt_dg = wt_info['dg'] if wt_info else self.raw_labels[idx].item()
        wt_emb = wt_info['emb'] if wt_info else self.embeddings[idx]

        return {
            'emb': self.embeddings[idx],
            'label_scaled': self.labels_scaled[idx],
            'wt_dg': torch.tensor([wt_dg]).float(),
            'wt_emb': wt_emb,
            'cluster_id': cluster_id
        }

## Protein VAE Model

In [7]:
class ProteinVAE(nn.Module):
    def __init__(self, input_dim=1280, latent_dim=256, hidden_dim=1024):
        super(ProteinVAE, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim * 2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
        self.regressor = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = h[:, :self.latent_dim], h[:, self.latent_dim:]
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        pred_y = self.regressor(z if self.training else mu)
        return recon, pred_y, mu, logvar

## Optimisation

In [8]:
def optimize_latent(model, emb, steps=25, lr=0.01):
    model.eval()

    with torch.no_grad():
        h = model.encoder(emb)
        z_start = h[:, :model.latent_dim].detach()

    z_opt = z_start.clone().requires_grad_(True)
    optimizer = optim.Adam([z_opt], lr=lr)

    for _ in range(steps):
        optimizer.zero_grad()
        pred_y = model.regressor(z_opt)
        loss = pred_y.mean() + 0.05 * torch.pow(z_opt, 2).mean()
        loss.backward()
        optimizer.step()

    return z_opt.detach()

## Training/Evaluation Pipelines

In [9]:
# --- 5. Training and Evaluation Pipeline ---
def train_and_validate(train_path, val_path, epochs=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_ds = ProteinDataset(train_path)
    val_ds = ProteinDataset(val_path, scaler=train_ds.scaler)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    model = ProteinVAE().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            x, y = batch['emb'].to(device), batch['label_scaled'].to(device)
            optimizer.zero_grad()
            recon, pred_y, mu, logvar = model(x)
            r_loss = F.mse_loss(recon, x)
            k_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
            p_loss = F.mse_loss(pred_y, y)
            (r_loss + 0.05 * k_loss + 1.0 * p_loss).backward()
            optimizer.step()

        if epoch % 10 == 0 or epoch == 1:
            model.eval()
            all_preds, all_gt, wt_baselines, nn_matched_dgs = [], [], [], []

            # --- PHASE 1: Metrics (No Grad) ---
            with torch.no_grad():
                for batch in val_loader:
                    emb, lbl = batch['emb'].to(device), batch['label_scaled'].to(device)
                    _, pred_y, _, _ = model(emb)
                    all_preds.extend(train_ds.scaler.inverse_transform(pred_y.cpu().numpy()).flatten())
                    all_gt.extend(train_ds.scaler.inverse_transform(lbl.cpu().numpy()).flatten())

            # --- PHASE 2: Discovery (Requires Grad for optimize_latent) ---
            batch_cluster_ids = batch['cluster_id']
            unique_clusters = np.unique(batch_cluster_ids)

            cluster_cache = {}
            with torch.no_grad():
                for c_id in unique_clusters:
                    l_idxs = val_ds.cluster_to_indices[c_id]
                    l_embs = val_ds.embeddings[l_idxs].to(device)

                    h_lib = model.encoder(l_embs)
                    recon_lib = model.decoder(h_lib[:, :model.latent_dim])

                    cluster_cache[c_id] = {
                        'recon': recon_lib,
                        'labels': val_ds.raw_labels[l_idxs]
                    }

            # 3. Optimize Latents (This is still slow because of backprop, but necessary for your logic)
            wt_dg, wt_emb = batch['wt_dg'].to(device), batch['wt_emb'].to(device)
            z_dream = optimize_latent(model, wt_emb)

            # 4. Perform Matching (Vectorized)
            with torch.no_grad():
                dream_recons = model.decoder(z_dream)

                for i in range(len(batch_cluster_ids)):
                    c_id = batch_cluster_ids[i]

                    lib_recon = cluster_cache[c_id]['recon']
                    lib_labels = cluster_cache[c_id]['labels']

                    dist = torch.norm(lib_recon - dream_recons[i].unsqueeze(0), dim=1, p=2)

                    nn_idx = torch.argmin(dist).item()

                    wt_baselines.append(wt_dg[i].item())
                    nn_matched_dgs.append(lib_labels[nn_idx].item())

            # --- Reporting ---
            all_preds, all_gt = np.array(all_preds), np.array(all_gt)
            wt_baselines, nn_matched_dgs = np.array(wt_baselines), np.array(nn_matched_dgs)

            mae = mean_absolute_error(all_gt, all_preds)
            rmse = np.sqrt(mean_squared_error(all_gt, all_preds))
            r2 = r2_score(all_gt, all_preds)
            rho, _ = spearmanr(all_gt, all_preds)
            pear, _ = pearsonr(all_gt, all_preds)

            success = np.sum(nn_matched_dgs < wt_baselines)

            print(f"\n--- Epoch {epoch:03d} ---")
            print(f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")
            print(f"Spearman: {rho:.4f} | Pearson: {pear:.4f}")
            print(f"Reconstructed Matching Success: {success}/{len(wt_baselines)} proteins better than WT.")

## Running the model
(Hyperparameter optimisation done here maybe)

In [ ]:
train_and_validate(train_path, val_path)